# 📊 Exploratory Data Analysis (EDA)
## Lightning Strikes & Flight Disruption Data Quality Assessment

**Date**: 1 avril 2026  
**Objective**: Analyze data quality, identify missing values, outliers, and correlations

---

## Section 1: Charger et Explorer les Données

Importer les bibliothèques nécessaires et charger les données du Data Lake

In [ ]:
# Import des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings

warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)

# Afficher plus de colonnes dans pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Bibliothèques importées avec succès!")

In [ ]:
# Charger les données depuis le Data Lake
data_lake_path = Path("../data/raw")

# Fonction pour charger les données JSON
def load_data_from_datalake():
    """Charger les données du Data Lake"""
    data_frames = []
    
    if data_lake_path.exists():
        json_files = list(data_lake_path.glob("*.json"))
        print(f"📁 Fichiers trouvés dans Data Lake: {len(json_files)}")
        
        for json_file in json_files[:5]:  # Charger les 5 premiers fichiers
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                    if isinstance(data, list):
                        df = pd.DataFrame(data)
                    else:
                        df = pd.DataFrame([data])
                    data_frames.append(df)
                    print(f"  ✓ Chargé: {json_file.name} ({len(df)} lignes)")
            except Exception as e:
                print(f"  ✗ Erreur pour {json_file.name}: {str(e)}")
    
    if data_frames:
        df_combined = pd.concat(data_frames, ignore_index=True)
        return df_combined
    else:
        # Si pas de fichiers, créer des données de démonstration
        print("⚠️  Création de données de démonstration...")
        np.random.seed(42)
        
        df_demo = pd.DataFrame({
            'lightning_id': [f'LIGHTNING_{i}' for i in range(200)],
            'latitude': np.random.uniform(40, 55, 200),
            'longitude': np.random.uniform(-10, 20, 200),
            'altitude': np.random.uniform(1000, 15000, 200),
            'intensity': np.random.uniform(10, 100, 200),
            'timestamp': pd.date_range('2026-01-01', periods=200, freq='h'),
            'source': np.random.choice(['api', 'web_scrape'], 200)
        })
        
        # Ajouter des valeurs manquantes
        missing_indices = np.random.choice(200, 30, replace=False)
        df_demo.loc[missing_indices, 'intensity'] = np.nan
        
        # Ajouter des valeurs aberrantes
        outlier_indices = np.random.choice(200, 10, replace=False)
        df_demo.loc[outlier_indices, 'intensity'] = np.random.uniform(500, 1000, 10)
        
        return df_demo

# Charger les données
df = load_data_from_datalake()

print(f"\n✅ Données chargées: {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\nAperçu des 5 premières lignes:")
print(df.head())

In [ ]:
# Informations de base sur le dataset
print("\n" + "="*70)
print("📋 INFOS GENERALES SUR LE DATASET")
print("="*70)

print(f"\n🎯 Dimensions:")
print(f"   - Lignes: {df.shape[0]}")
print(f"   - Colonnes: {df.shape[1]}")

print(f"\n📊 Types de données:")
print(df.dtypes)

print(f"\n📈 Statistiques descriptives (colonnes numériques):")
print(df.describe().round(2))

print(f"\n⚠️  Aperçu de la qualité des données:")
print(df.info())

## Section 2: Calculer le Pourcentage de Données Manquantes

Analyser les valeurs manquantes (NaN) pour chaque colonne

In [ ]:
# Calculer les données manquantes
missing_data = pd.DataFrame({
    'Colonne': df.columns,
    'Valeurs Manquantes': [df[col].isna().sum() for col in df.columns],
    'Nombre Total': [len(df)] * len(df.columns)
})

missing_data['% Manquantes'] = (missing_data['Valeurs Manquantes'] / missing_data['Nombre Total'] * 100).round(2)
missing_data = missing_data.sort_values('% Manquantes', ascending=False)

print("\n" + "="*70)
print("🔍 DONNÉES MANQUANTES PAR COLONNE")
print("="*70)
print(missing_data.to_string(index=False))

# Résumé global
total_missing = df.isna().sum().sum()
total_cells = df.shape[0] * df.shape[1]
overall_missing_pct = (total_missing / total_cells * 100)

print(f"\n📊 RÉSUMÉ GLOBAL:")
print(f"   - Total de valeurs manquantes: {total_missing}")
print(f"   - Total de cellules: {total_cells}")
print(f"   - Pourcentage global manquant: {overall_missing_pct:.2f}%")

In [ ]:
# Visualisation des données manquantes
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graphique 1: Pourcentage manquant par colonne
missing_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=True)
if (missing_pct > 0).any():
    missing_pct[missing_pct > 0].plot(kind='barh', ax=axes[0], color='coral')
    axes[0].set_xlabel('Pourcentage (%)', fontsize=12, fontweight='bold')
    axes[0].set_title('📊 Pourcentage de Données Manquantes par Colonne', fontsize=13, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)
    
    # Ajouter les valeurs sur les barres
    for i, v in enumerate(missing_pct[missing_pct > 0]):
        axes[0].text(v + 0.1, i, f'{v:.1f}%', va='center', fontweight='bold')
else:
    axes[0].text(0.5, 0.5, 'Aucune donnée manquante', 
                 ha='center', va='center', transform=axes[0].transAxes, fontsize=14)
    axes[0].set_title('📊 Pourcentage de Données Manquantes par Colonne', fontsize=13, fontweight='bold')

# Graphique 2: Visualisation de la complétude
completeness = ((df.isna().sum() / len(df)) * 100).sort_values(ascending=False)
colors = ['#ff6b6b' if x > 0 else '#51cf66' for x in completeness]
completeness.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_ylabel('% de Complétude', fontsize=12, fontweight='bold')
axes[1].set_title('✅ Niveau de Complétude des Données', fontsize=13, fontweight='bold')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_ylim([0, 105])
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Visualisations générées!")

## Section 3: Identifier et Quantifier les Valeurs Aberrantes

Détecter les valeurs aberrantes (outliers) en utilisant la méthode IQR (Interquartile Range)

In [ ]:
# Fonction pour détecter les valeurs aberrantes (méthode IQR)
def detect_outliers_iqr(data):
    """
    Détecter les valeurs aberrantes avec la méthode IQR
    """
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    return (data < lower_bound) | (data > upper_bound)

# Détecter les valeurs aberrantes pour chaque colonne numérique
numeric_columns = df.select_dtypes(include=[np.number]).columns

outliers_summary = []

print("\n" + "="*70)
print("🔴 DÉTECTION DES VALEURS ABERRANTES (Méthode IQR)")
print("="*70)

for col in numeric_columns:
    if df[col].notna().sum() > 0:  # Vérifier qu'il y a des données non manquantes
        outliers_mask = detect_outliers_iqr(df[col].dropna())
        outlier_count = outliers_mask.sum()
        outlier_pct = (outlier_count / len(df[col].dropna()) * 100) if df[col].notna().sum() > 0 else 0
        
        outliers_summary.append({
            'Colonne': col,
            'Valeurs Aberrantes': outlier_count,
            'Total Non-Vides': df[col].notna().sum(),
            '% Aberrantes': round(outlier_pct, 2)
        })

outliers_df = pd.DataFrame(outliers_summary).sort_values('% Aberrantes', ascending=False)
print(outliers_df.to_string(index=False))

# Résumé global des aberrantes
total_outliers = outliers_df['Valeurs Aberrantes'].sum()
total_numeric = df[numeric_columns].notna().sum().sum()

print(f"\n📊 RÉSUMÉ GLOBAL DES ABERRANTES:")
print(f"   - Total d'observations numériques: {total_numeric}")
print(f"   - Total de valeurs aberrantes: {total_outliers}")
print(f"   - Pourcentage global aberrant: {(total_outliers / total_numeric * 100):.2f}%")

In [ ]:
# Boxplots pour visualiser les aberrantes
if len(numeric_columns) > 0:
    n_cols = min(3, len(numeric_columns))
    n_rows = (len(numeric_columns) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5*n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, col in enumerate(numeric_columns):
        ax = axes[idx]
        
        # Boxplot
        bp = ax.boxplot([df[col].dropna()], vert=True, patch_artist=True,
                         boxprops=dict(facecolor='lightblue', color='navy'),
                         whiskerprops=dict(color='navy'),
                         capprops=dict(color='navy'),
                         medianprops=dict(color='red', linewidth=2),
                         flierprops=dict(marker='o', markerfacecolor='red', markersize=8, alpha=0.5))
        
        ax.set_ylabel('Valeur', fontsize=11, fontweight='bold')
        ax.set_title(f'📦 Boxplot: {col}', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        
        # Statistiques sur la boîte
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        median = df[col].median()
        IQR = Q3 - Q1
        
        stats_text = f'Q1={Q1:.2f}\nMédian={median:.2f}\nQ3={Q3:.2f}\nIQR={IQR:.2f}'
        ax.text(1.15, median, stats_text, fontsize=9, 
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Masquer les subplots inutilisés
    for idx in range(len(numeric_columns), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    print("✅ Boxplots générés!")
else:
    print("⚠️  Aucune colonne numérique pour créer les boxplots")

## Section 4: Générer la Matrice de Corrélation

Calculer les corrélations entre les variables numériques

In [ ]:
# Calculer la matrice de corrélation (Pearson)
if len(numeric_columns) > 1:
    # Créer une copie du dataframe avec seulement les colonnes numériques
    df_numeric = df[numeric_columns].copy()
    
    # Calculer la matrice de corrélation
    correlation_matrix = df_numeric.corr(method='pearson')
    
    print("\n" + "="*70)
    print("🔗 MATRICE DE CORRÉLATION DE PEARSON")
    print("="*70)
    print("\nValeurs de corrélation (arrondi à 3 décimales):")
    print(correlation_matrix.round(3))
    
    # Identifier les corrélations fortes
    print("\n📊 CORRÉLATIONS FORTES (|r| > 0.7):")
    strong_corr = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            corr_val = correlation_matrix.iloc[i, j]
            if abs(corr_val) > 0.7:
                strong_corr.append({
                    'Variable 1': correlation_matrix.columns[i],
                    'Variable 2': correlation_matrix.columns[j],
                    'Corrélation': round(corr_val, 3)
                })
    
    if strong_corr:
        strong_corr_df = pd.DataFrame(strong_corr)
        print(strong_corr_df.to_string(index=False))
    else:
        print("   Aucune corrélation forte détectée")
    
else:
    print("⚠️  Moins de 2 colonnes numériques pour calculer la corrélation")

## Section 5: Visualiser la Matrice de Corrélation

Créer une heatmap pour identifier visuellement les relations entre variables

In [ ]:
# Créer la heatmap de la matrice de corrélation
if len(numeric_columns) > 1:
    plt.figure(figsize=(12, 10))
    
    # Créer la heatmap
    sns.heatmap(correlation_matrix, 
                annot=True,               # Afficher les valeurs
                cmap='RdBu_r',            # Couleurs (rouge/bleu)
                center=0,                 # Centrer sur 0
                vmin=-1, vmax=1,          # Limites de valeurs
                fmt='.2f',                # Format des nombres
                square=True,              # Carrés
                linewidths=0.5,           # Lignes entre les cases
                cbar_kws={"shrink": 0.8}, # Barre de couleur
                annot_kws={"fontsize": 10, "fontweight": "bold"})
    
    plt.title('🔥 Heatmap de Corrélation - Matrice de Pearson', 
              fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Variables', fontsize=12, fontweight='bold')
    plt.ylabel('Variables', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Heatmap de corrélation générée!")
else:
    print("⚠️  Impossible de créer la heatmap avec moins de 2 colonnes numériques")

## Section 6: Créer des Diagrammes d'Analyse

Générer des visualisations supplémentaires : histogrammes, distributions, et diagrammes en barres

In [ ]:
# Histogrammes pour les distributions
if len(numeric_columns) > 0:
    n_cols = min(3, len(numeric_columns))
    n_rows = (len(numeric_columns) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5*n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, col in enumerate(numeric_columns):
        ax = axes[idx]
        
        # Histogramme
        data_to_plot = df[col].dropna()
        
        ax.hist(data_to_plot, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
        ax.axvline(data_to_plot.mean(), color='red', linestyle='--', linewidth=2, label=f'Moyenne: {data_to_plot.mean():.2f}')
        ax.axvline(data_to_plot.median(), color='green', linestyle='--', linewidth=2, label=f'Médiane: {data_to_plot.median():.2f}')
        
        ax.set_xlabel('Valeur', fontsize=11, fontweight='bold')
        ax.set_ylabel('Fréquence', fontsize=11, fontweight='bold')
        ax.set_title(f'📊 Distribution: {col}', fontsize=12, fontweight='bold')
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
    
    # Masquer les subplots inutilisés
    for idx in range(len(numeric_columns), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    print("✅ Histogrammes générés!")
else:
    print("⚠️  Aucune colonne numérique pour créer les histogrammes")

In [ ]:
# Diagramme comparatif : Données manquantes vs Aberrantes
comparison_data = []

for col in numeric_columns:
    missing_count = df[col].isna().sum()
    missing_pct = (missing_count / len(df) * 100)
    
    # Valeurs aberrantes
    outlier_mask = detect_outliers_iqr(df[col].dropna())
    outlier_count = outlier_mask.sum()
    outlier_pct = (outlier_count / len(df[col].dropna()) * 100) if df[col].notna().sum() > 0 else 0
    
    comparison_data.append({
        'Colonne': col,
        'Manquantes (%)': missing_pct,
        'Aberrantes (%)': outlier_pct
    })

comparison_df = pd.DataFrame(comparison_data)

# Créer le diagramme en barres groupées
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df['Manquantes (%)'], width, 
               label='Données Manquantes', color='#FF6B6B', alpha=0.8)
bars2 = ax.bar(x + width/2, comparison_df['Aberrantes (%)'], width, 
               label='Valeurs Aberrantes', color='#FFA500', alpha=0.8)

ax.set_xlabel('Colonnes', fontsize=12, fontweight='bold')
ax.set_ylabel('Pourcentage (%)', fontsize=12, fontweight='bold')
ax.set_title('📊 Comparaison: Données Manquantes vs Aberrantes', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Colonne'], rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Ajouter les valeurs sur les barres
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Diagramme comparatif généré!")

## Section 7: Résumé et Recommandations

Analyse finale et recommandations pour le nettoyage des données

In [ ]:
# Résumé global et recommandations
print("\n" + "="*70)
print("📋 RÉSUMÉ GLOBAL DE L'ANALYSE EDA")
print("="*70)

print(f"\n🎯 DIMENSIONS DES DONNÉES:")
print(f"   - Nombre de lignes: {df.shape[0]}")
print(f"   - Nombre de colonnes: {df.shape[1]}")
print(f"   - Colonnes numériques: {len(numeric_columns)}")

print(f"\n⚠️  QUALITÉ DES DONNÉES:")
print(f"   - Pourcentage global manquant: {overall_missing_pct:.2f}%")
print(f"   - Pourcentage global aberrant: {(total_outliers / total_numeric * 100):.2f}%")

print(f"\n📊 COLONNES AVEC DONNÉES MANQUANTES:")
missing_cols = missing_data[missing_data['% Manquantes'] > 0]
if len(missing_cols) > 0:
    for _, row in missing_cols.iterrows():
        print(f"   • {row['Colonne']}: {row['% Manquantes']:.2f}%")
else:
    print("   ✓ Aucune donnée manquante")

print(f"\n📊 COLONNES AVEC VALEURS ABERRANTES:")
outlier_cols = outliers_df[outliers_df['% Aberrantes'] > 0]
if len(outlier_cols) > 0:
    for _, row in outlier_cols.iterrows():
        print(f"   • {row['Colonne']}: {row['% Aberrantes']:.2f}%")
else:
    print("   ✓ Aucune valeur aberrante détectée")

print(f"\n💡 RECOMMANDATIONS:")
print(f"""
   1. ✅ IMPUTATION DES DONNÉES MANQUANTES:
      • Pour les colonnes avec < 5% manquant: interpolation ou moyenne/médiane
      • Pour les colonnes avec > 20% manquant: considérer la suppression
      
   2. ✅ TRAITEMENT DES ABERRANTES:
      • Vérifier la validité des aberrantes (erreurs ou valeurs réelles)
      • Options: suppression, cap/floor, ou transformation
      
   3. ✅ NORMALISATION:
      • Vérifier les distributions pour décider de la normalisation
      • Utiliser StandardScaler ou MinMaxScaler selon le besoin
      
   4. ✅ NETTOYAGE:
      • Supprimer les colonnes avec > 50% manquant
      • Valider les types de données
      
   5. ✅ VALIDATION:
      • Refaire cette analyse après nettoyage
      • Comparer avant/après pour valider les modifications
""")

print("\n" + "="*70)
print("✅ ANALYSE EDA TERMINÉE!")
print("="*70)

In [ ]:
# Créer un rapport détaillé en PDF (optionnel)
print("\n" + "="*70)
print("📄 EXPORTER LES RÉSULTATS")
print("="*70)

# Créer un dictionnaire avec tous les résultats
eda_report = {
    'Dataset Dimensions': {
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Numeric Columns': len(numeric_columns)
    },
    'Data Quality': {
        'Overall Missing (%)': round(overall_missing_pct, 2),
        'Overall Outliers (%)': round((total_outliers / total_numeric * 100) if total_numeric > 0 else 0, 2)
    },
    'Missing Data Summary': missing_data.to_dict('records'),
    'Outliers Summary': outliers_df.to_dict('records'),
}

print("\n✅ Résultats de l'EDA disponibles dans le notebook")
print("   - Graphiques de qualité des données")
print("   - Matrice de corrélation (heatmap)")
print("   - Histogrammes de distributions")
print("   - Analyse des valeurs aberrantes")
print("   - Recommandations de nettoyage")

print("\n💾 Pour sauvegarder les résultats:")
print("   • Exporter le notebook: File > Download as > PDF / HTML")
print("   • Ou générer un rapport personnalisé avec les graphiques")